In [ ]:
# %% [markdown]
# # Support Integrity Auditor (SIA) — Full Reproducible Pipeline
# ### Stages: Pseudo-labeling → Feature Engineering → Training → Inference → Dossiers

# %% [markdown]
# ## 0. Setup & Imports

# %%
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")
Path("models/").mkdir(exist_ok=True)
Path("results/").mkdir(exist_ok=True)
Path("data/").mkdir(exist_ok=True)

# %% [markdown]
# ## 1. Data Loading & Preprocessing

# %%
from train_pipeline import load_and_preprocess
df = load_and_preprocess("data/tickets.csv")
print(df.head())
print("\nShape:", df.shape)
print("\nPriority distribution:")
print(df["ticket_priority"].value_counts())

# %% [markdown]
# ## 2. Pseudo-Label Generation (Self-Supervised)

# %%
from train_pipeline import generate_pseudo_labels
df, embeddings = generate_pseudo_labels(df)

print(f"\nMismatch rate: {df['mismatch_label'].mean():.2%}")
df[["ticket_id","ticket_priority","inferred_severity","mismatch_type","fused_score"]].head(10)

# %%
# Visualize signal distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes,
    ["sig1_nlp", "sig2_embed", "sig3_restime"],
    ["Signal 1: NLP Rule-based", "Signal 2: Embedding Cluster", "Signal 3: Resolution Time"]):
    ax.hist(df[col], bins=30, color="steelblue", edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("results/signal_distributions.png", dpi=150)
plt.show()

# %% [markdown]
# ## 3. Signal Agreement Analysis

# %%
# Pairwise signal agreement
for s1, s2 in [("sig1_nlp","sig2_embed"),("sig1_nlp","sig3_restime"),("sig2_embed","sig3_restime")]:
    b1 = (df[s1] >= 0.5).astype(int)
    b2 = (df[s2] >= 0.5).astype(int)
    agreement = (b1 == b2).mean()
    print(f"Agreement {s1} ↔ {s2}: {agreement:.4f}")

# %% [markdown]
# ## 4. Ablation Study

# %%
from train_pipeline import run_ablation, engineer_features
X, y, df = engineer_features(df)
ablation = run_ablation(df, y)

import pandas as pd
abl_df = pd.DataFrame(ablation).T
print("\nAblation Results:")
print(abl_df.to_string())

# %% [markdown]
# ## 5. Classifier Training

# %%
from train_pipeline import train_classifier
results = train_classifier(X, y)
metrics = results["metrics"]
print(f"\nFinal Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

# %% [markdown]
# ## 6. Dossier Generation

# %%
from train_pipeline import generate_all_dossiers
dossiers = generate_all_dossiers(df)
print(f"\nSample Dossier:")
print(json.dumps(dossiers[0], indent=2))

# %% [markdown]
# ## 7. Verification Check

# %%
thresholds = {"accuracy": 0.83, "macro_f1": 0.82,
              "recall_consistent": 0.78, "recall_mismatch": 0.78}

print("="*50)
print("VERIFICATION REPORT")
print("="*50)
all_pass = True
for k, thresh in thresholds.items():
    val = metrics.get(k, 0)
    status = "✅ PASS" if val >= thresh else "❌ FAIL"
    if val < thresh:
        all_pass = False
    print(f"  {k:<25}: {val:.4f} (≥{thresh}) → {status}")
print("="*50)
print("OVERALL:", "✅ ALL CRITERIA MET" if all_pass else "❌ CRITERIA NOT MET")

# %% [markdown]
# ## 8. Adversarial Test

# %%
from predict import predict_single, load_models
from app import ADVERSARIAL_TICKETS

load_models()
score = 0
for t in ADVERSARIAL_TICKETS:
    ticket = {
        "ticket_id": t["id"],
        "ticket_subject": t["subject"],
        "ticket_description": t["description"],
        "ticket_priority": t["priority"],
        "ticket_channel": "email",
        "ticket_type": "Technical Issue",
        "product_purchased": "Enterprise",
        "resolution_time": 24.0,
        "customer_email": "test@test.com"
    }
    r = predict_single(ticket)
    expected_mismatch = "Mismatch" in t["expected"]
    correct = (r["mismatch_label"] == 1) == expected_mismatch
    score += int(correct)
    icon = "✅" if correct else "❌"
    print(f"{icon} {t['id']}: Pred={r['mismatch_type']} | Expected={t['expected']}")

print(f"\nAdversarial Score: {score}/10 | {'BONUS EARNED' if score >= 7 else 'No bonus'}")